In [1]:
from pyspark.sql.functions import col, to_date, col, lit, when, substring
from pyspark.sql.functions import lower, regexp_replace, trim
from pyspark.sql.types import StructType, StructField, StringType


StatementMeta(, eef16d12-2c42-41d8-9688-395fb6d29d5b, 3, Finished, Available, Finished, False)

In [2]:
# Load the five source CSV files into separate Spark DataFrames.
df1 = spark.read.format("csv")\
.option("header","true")\
.load("Files/raw/api_data_aadhar_demographic/api_data_aadhar_demographic_0_500000.csv")

df2 = spark.read.format("csv").option("header","true").load("Files/raw/api_data_aadhar_demographic/api_data_aadhar_demographic_1000000_1500000.csv")

df3 = spark.read.format("csv").option("header","true").load("Files/raw/api_data_aadhar_demographic/api_data_aadhar_demographic_1500000_2000000.csv")

df4 = spark.read.format("csv").option("header","true").load("Files/raw/api_data_aadhar_demographic/api_data_aadhar_demographic_2000000_2071700.csv")

df5 = spark.read.format("csv").option("header","true").load("Files/raw/api_data_aadhar_demographic/api_data_aadhar_demographic_500000_1000000.csv")

# Preview the first few records to confirm that the data was loaded correctly.
display(df1.limit(5))


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 571e4d6b-9326-4714-b0cd-72c529e30eee)

In [3]:
demo_df = df1.union(df2).union(df3).union(df4).union(df5)

# Display a sample of the combined DataFrame.
display(demo_df.limit(5))


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 85c58806-51da-45c2-af9f-34be79abac3f)

In [4]:
print(f"df schema : {demo_df.printSchema()}")
print(f"df describe : {demo_df.describe()}")
print(f"df rowCount : {demo_df.count()}")

StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 6, Finished, Available, Finished, False)

root
 |-- date: string (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- pincode: string (nullable = true)
 |-- demo_age_5_17: string (nullable = true)
 |-- demo_age_17_: string (nullable = true)

df schema : None
df describe : DataFrame[summary: string, date: string, state: string, district: string, pincode: string, demo_age_5_17: string, demo_age_17_: string]
df rowCount : 2071700


In [5]:
# Apply the required data type transformations to the demographic dataset.
demo_df = demo_df.select(
    # Convert the date string into a Spark date type.
    to_date(col("date") , "dd-MM-yyyy").alias("date"),
    
    # Keep location fields as strings.
    col("state").cast("string"),
    col("district").cast("string"),
    
    # Convert pincode and age-group counts to integer values.
    col("pincode").cast("integer"),
    col("demo_age_5_17").cast("integer"),
    col("demo_age_17_").cast("integer")
)

# Verify that the transformations produced the expected schema.
demo_df.printSchema()


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 7, Finished, Available, Finished, False)

root
 |-- date: date (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- pincode: integer (nullable = true)
 |-- demo_age_5_17: integer (nullable = true)
 |-- demo_age_17_: integer (nullable = true)



In [6]:
demo_df.select("state").distinct().show()

StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 8, Finished, Available, Finished, False)

+--------------------+
|               state|
+--------------------+
|            Nagaland|
|           Karnataka|
|              Odisha|
|              Kerala|
|         WEST BENGAL|
|              Ladakh|
|Dadra and Nagar H...|
|          Tamil Nadu|
|              odisha|
|        Chhattisgarh|
|      Andhra Pradesh|
|         west Bengal|
|         Lakshadweep|
|      Madhya Pradesh|
|              Punjab|
|             Manipur|
|         Daman & Diu|
|     Jammu & Kashmir|
|                 Goa|
|      andhra pradesh|
+--------------------+
only showing top 20 rows



In [7]:
# Inspect records where the state contains the invalid value "100000".
demo_df.filter(col("state") == "100000").show()

# Count the number of records containing the invalid state value.
count = demo_df.filter(col("state") == "100000").count()
print(f"Total fields with incorrect data: {count}")

# Remove records containing the invalid state value.
demo_df = demo_df.filter(col("state") != "100000")


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 9, Finished, Available, Finished, False)

+----------+------+--------+-------+-------------+------------+
|      date| state|district|pincode|demo_age_5_17|demo_age_17_|
+----------+------+--------+-------+-------------+------------+
|2025-12-20|100000|  100000| 100000|            0|           1|
|2025-12-23|100000|  100000| 100000|            0|           1|
+----------+------+--------+-------+-------------+------------+

Total fields with incorrect data: 2


In [8]:
def normalize_text(col_obj):
    # Convert text to lowercase so that comparisons are case-insensitive.
    c = lower(col_obj)

    # Standardise the ampersand character.
    c = regexp_replace(c, "&", "and")

    # Replace spaces, dots, and hyphens with underscores.
    c = regexp_replace(c, r"[\s\.\-]+", "_")

    # Collapse consecutive underscores into a single underscore.
    c = regexp_replace(c, r"_+", "_")

    # Remove "the_" when it appears at the beginning of a value.
    c = regexp_replace(c, r"^the_", "")

    # Remove unwanted underscores and asterisks from the beginning and end.
    c = regexp_replace(c, r"^[_*]+", "")
    c = regexp_replace(c, r"[_*]+$", "")
    
    return trim(c)


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 10, Finished, Available, Finished, False)

In [9]:
demo_df = demo_df \
    .withColumnRenamed("state", "oldState") \
    .withColumnRenamed("district", "oldDistrict") \
    .withColumn("state", normalize_text(col("oldState"))) \
    .withColumn("district", normalize_text(col("oldDistrict")))

# Verify the resulting schema and inspect sample records.
demo_df.printSchema()
display(demo_df.limit(10))


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 11, Finished, Available, Finished, False)

root
 |-- date: date (nullable = true)
 |-- oldState: string (nullable = true)
 |-- oldDistrict: string (nullable = true)
 |-- pincode: integer (nullable = true)
 |-- demo_age_5_17: integer (nullable = true)
 |-- demo_age_17_: integer (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 4a1b3e84-b817-4d59-a952-714f644254c3)

In [10]:
states_df = spark.read.format("csv").option("header","true").load("Files/raw/states/STATES.csv")

# Normalise location names and convert pincode to an integer.
states_df = states_df \
    .withColumn("state", normalize_text(col("statename"))) \
    .withColumn("district", normalize_text(col("district"))) \
    .withColumn("pincode", col("pincode").cast("integer")) \
    .select("state", "district", "pincode") \
    .filter(col("state").isNotNull() & (col("state") != "na"))

# Display the distinct state names available in the reference dataset.
display(states_df.select("state").distinct())


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f95e10f4-7f0f-46e1-b59e-30eb646d043e)

In [11]:
missing_states = demo_df.join(
    states_df, 
    on="state", 
    how="left_anti"
).select("state").distinct()

display(missing_states)


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3c67cef7-e020-4f6a-aa04-52c6fd664d05)

In [12]:
from pyspark.sql import functions as F

demo_df = demo_df.withColumn(
    "state", 
    F.when(F.col("state") == "pondicherry", "puducherry")
    .when(F.col("state") == "orissa", "odisha")
    .when(F.col("state").isin("westbengal", "west_bangal", "west_bengli"), "west_bengal")
    .when(F.col("state").isin("dadra_and_nagar_haveli", "daman_and_diu"), "dadra_and_nagar_haveli_and_daman_and_diu")
    .when(F.col("state") == "chhatisgarh", "chhattisgarh")
    .when(F.col("state") == "uttaranchal", "uttarakhand")
    # Mapping specific city/locality names to their respective states if required
    .when(F.col("state") == "darbhanga", "bihar")
    .when(F.col("state") == "puttenahalli", "karnataka")
    .when(F.col("state") == "balanagar", "telangana")
    .when(F.col("state") == "madanapalle", "andhra_pradesh")
    .when(F.col("state") == "jaipur", "rajasthan")
    .when(F.col("state") == "nagpur", "maharashtra")
    .when(F.col("state") == "raja_annamalai_puram", "tamil_nadu")
    .otherwise(F.col("state"))
)


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 14, Finished, Available, Finished, False)

In [13]:
missing_states = demo_df.join(
    states_df, 
    on="state", 
    how="left_anti"
).select("state").distinct()

missing_states.show()


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 15, Finished, Available, Finished, False)

+-----+
|state|
+-----+
+-----+



In [14]:
# Validate the state-district combination against the trusted reference data.
missing_districts = demo_df.join(
    states_df.select("state", "district").distinct(),
    on=["state", "district"],
    how="left_anti"
).select("state", "district").distinct()

print(f"Number of unmatched state-district combinations: {missing_districts.count()}")


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 16, Finished, Available, Finished, False)

Number of unmatched state-district combinations: 273


In [15]:
import pyspark.sql.functions as F

# Identify records whose state-district combination does not exist
# in the trusted reference dataset.
invalid_location_df = demo_df.join(
    states_df.select("state", "district").distinct(),
    on=["state", "district"],
    how="left_anti"
).select(
    "state",
    "district"
).distinct() \
 .withColumn("is_invalid_location", F.lit(True))


# Create a pincode-based reference lookup containing the trusted
# state and district values.
#
# dropDuplicates() prevents duplicate reference records from
# multiplying rows when the lookup is joined to the demographic data.
pincode_lookup_df = states_df.select(
    "pincode",
    F.col("state").alias("correct_state"),
    F.col("district").alias("correct_district")
).dropDuplicates(["pincode"])


# Attach the invalid-location flag and the reference location
# associated with each pincode.
updated_df = demo_df.join(
    invalid_location_df,
    on=["state", "district"],
    how="left"
).join(
    pincode_lookup_df,
    on="pincode",
    how="left"
)


# Replace the state and district only when the original
# state-district combination is invalid and a valid pincode
# reference value is available.
demo_df = updated_df.withColumn(
    "district",
    F.when(
        F.col("is_invalid_location").isNotNull() &
        F.col("correct_district").isNotNull(),
        F.col("correct_district")
    ).otherwise(F.col("district"))
).withColumn(
    "state",
    F.when(
        F.col("is_invalid_location").isNotNull() &
        F.col("correct_state").isNotNull(),
        F.col("correct_state")
    ).otherwise(F.col("state"))
).drop(
    "is_invalid_location",
    "correct_state",
    "correct_district"
)


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 17, Finished, Available, Finished, False)

In [16]:
district_mapping = {
    "andhra_pradesh": {
        "khammam": {"telangana": "khammam"},
        "nalgonda": {"telangana": "nalgonda"},
        "sri_potti_sriramulu_nellore": {"andhra_pradesh": "spsr_nellore"},
        "nellore": {"andhra_pradesh": "spsr_nellore"},
        "hyderabad": {"telangana": "hyderabad"},
        "k_v_rangareddy": {"telangana": "ranga_reddy"},
        "rangareddi": {"telangana": "ranga_reddy"},
        "mahabubnagar": {"telangana" : "mahabubnagar"},
        "mahbubnagar": {"telangana" : "mahabubnagar"},
        "visakhapatnam" : {"andhra_pradesh":"visakhapatanam"}
    },
    "bihar": {
        "west_champaran": {"bihar": "pashchim_champaran"},
        "east_champaran": {"bihar": "purbi_champaran"},
        "purba_champaran": {"bihar": "purbi_champaran"},
        "purnea" : {"bihar": "purnia"},
        "samstipur" :{"bihar":"samastipur"},
        "monghyr" : {"bihar" : "munger"},
        "aurangabad(bh)" : {"bihar" : "aurangabad"}
    },
    "chhattisgarh": {
        "dakshin_bastar_dantewada": {"chhattisgarh": "dantewada"},
    },
    "gujarat": {
        "ahmedabad": {"gujarat": "ahmadabad"},
        "panchmahals": {"gujarat": "panch_mahals"},
        "surendra_nagar": {"gujarat": "surendranagar"},
    },
    "haryana": {
        "yamuna_nagar": {"haryana": "yamunanagar"},
    },
    "jammu_and_kashmir": {
        "punch": {"jammu_and_kashmir": "poonch"},
        "baramula": {"jammu_and_kashmir": "baramulla"},
        "kargil": {"ladakh": "kargil"},
        "leh": {"ladakh": "leh_ladakh"}
    },
    "jharkhand": {
        "purbi_singhbhum": {"jharkhand": "east_singhbum"},
        "pashchimi_singhbhum": {"jharkhand": "west_singhbhum"},
        "east_singhbhum": {"jharkhand": "east_singhbum"},
        "seraikela_kharsawan":{"jharkhand" : "saraikela_kharsawan"}
    },
    "karnataka": {
        "chickmagalur": {"karnataka": "chikkamagaluru"},
        "hasan": {"karnataka": "hassan"},
        "bijapur": {"karnataka": "vijayapura"},
        "shimoga": {"karnataka": "shivamogga"},
        "mysore": {"karnataka": "mysuru"},
        "belgaum": {"karnataka": "belagavi"},
        "tumkur": {"karnataka": "tumakuru"},
        "ramanagar": {"karnataka": "ramanagara"},
        "chikmagalur" : {"karnataka":"chikkamagaluru"},
        "bangalore_rural" : {"karnataka":"bengaluru_rural"}
    },
    "ladakh": {
        "leh": {"ladakh": "leh_ladakh"},
    },
    "madhya_pradesh": {
        "narsimhapur": {"madhya_pradesh": "narsinghpur"},
        "ashok_nagar" : {"madhya_pradesh":"ashoknagar"}
    },
    "maharashtra": {
        "ahmadnagar": {"maharashtra": "ahmednagar"},
        "mumbai(_sub_urban_)": {"maharashtra": "mumbai_suburban"},
        "mumbai(_sub_urban_)": {"maharashtra": "mumbai_suburban"},
        "mumbai_city": {"maharashtra": "mumbai"},
        "ahilyanagar" : {"maharashtra":"ahmednagar"},
        "ahmed_nagar" : {"maharashtra" : "ahmednagar"}
    },
    "mizoram": {
        "mammit": {"mizoram": "mamit"},
    },
    "odisha": {
        "angul": {"odisha": "anugul"},
        "subarnapur": {"odisha": "sonepur"},
        "baleswar": {"odisha": "baleshwar"},
        "balasore" : {"odisha": "baleshwar"},
        "jagatsinghpur" : {"odisha":"jagatsinghapur"}
    },
    "puducherry": {
        "puducherry": {"puducherry": "pondicherry"},
    },
    "punjab": {
        "sas_nagar_(mohali)": {"punjab": "s_a_s_nagar"},
        "sas_nagar" : {"punjab": "s_a_s_nagar"},
        "firozpur" : {"punjab":"firozepur"},
        "shaheed_bhagat_singh_nagar" :{"punjab" : "shahid_bhagat_singh_nagar"}
    },
    "rajasthan": {
        "jhunjhunun": {"rajasthan": "jhunjhunu"},
        "chittaurgarh": {"rajasthan": "chittorgarh"},
        "didwana_kuchaman": {"maharashtra":"nagpur"},
        "khairthal_tijara":{"rajasthan" : "alwar"}
    },
    "sikkim": {
        "east_sikkim": {"sikkim": "east_district"},
        "east": {"sikkim": "east_district"},
        "sikkim": {"sikkim": "east_district"},
        "north_sikkim": {"sikkim": "north_district"},
        "north": {"sikkim": "north_district"},
        "gangtok" : {"sikkim" : "east_district"},
        "mangan" : {"sikkim" : "north_district"}
    },
    "tamil_nadu": {
        "kancheepuram": {"tamil_nadu": "kanchipuram"},
        "kanyakumari": {"tamil_nadu": "kanniyakumari"},
        "tiruvallur": {"tamil_nadu": "thiruvallur"},
        "thoothukkudi" : {"tamil_nadu":"tuticorin"}
    },
    "telangana": {
        "k_v_rangareddy": {"telangana": "ranga_reddy"},
        "rangareddy": {"telangana": "ranga_reddy"},
    },
    "uttar_pradesh": {
        "allahabad": {"uttar_pradesh": "prayagraj"},
        "bara_banki": {"uttar_pradesh": "barabanki"},
        "bulandshahar": {"uttar_pradesh": "bulandshahr"},
        "sant_kabir_nagar": {"uttar_pradesh": "sant_kabeer_nagar"},
        "faizabad": {"uttar_pradesh": "ayodhya"},
        "sant_ravidas_nagar" : {"uttar_pradesh":"bhadohi"},
        "bagpat" : {"uttar_pradesh":"baghpat"}
    },
    "uttarakhand": {
        "udham_singh_nagar": {"uttarakhand": "udam_singh_nagar"},
        "garhwal" : {"uttarakhand" : "tehri_garhwal"}
    },
    "west_bengal": {
        "dakshin_dinajpur" : {"west_bengal": "dinajpur_dakshin"},
        "paschim_medinipur": {"west_bengal": "medinipur_west"},
        "north_dinajpur": {"west_bengal": "dinajpur_uttar"},
        "bardhaman": {"west_bengal": "purba_bardhaman"},
        "malda": {"west_bengal": "maldah"},
        "north_24_parganas": {"west_bengal": "24_paraganas_north"},
        "south_twenty_four_parganas": {"west_bengal": "24_paraganas_south"},
        "puruliya": {"west_bengal": "purulia"},
        "cooch_behar": {"west_bengal": "coochbehar"},
        "purba_medinipur": {"west_bengal": "medinipur_east"},
        "koch_bihar": {"west_bengal": "coochbehar"},
        "haora": {"west_bengal": "howrah"},
        "dakshin_dinajpur": {"west_bengal": "dinajpur_dakshin"},
        "barddhaman": {"west_bengal": "purba_bardhaman"},
        "south_24_parganas": {"west_bengal": "24_paraganas_south"},
        "uttar_dinajpur": {"west_bengal": "dinajpur_uttar"},
        "south_dinajpur": {"west_bengal": "dinajpur_dakshin"},
        "medinipur": {"west_bengal": "medinipur_west"},
        "north_twenty_four_parganas": {"west_bengal": "24_paraganas_south"},
        "east_midnapore": {"west_bengal": "medinipur_east"},
        "darjiling": {"west_bengal": "darjeeling"},
        "west_midnapore": {"west_bengal": "medinipur_west"},
        "hugli": {"west_bengal": "hooghly"},
        
    },
}


from pyspark.sql import functions as F

# Build mapping:
# "source_state\tsource_district" -> "target_state\ttarget_district"
mapping_expr = F.create_map(
    *[
        F.lit(item)
        for state, districts in district_mapping.items()
        for district, target in districts.items()
        for target_state, target_district in target.items()
        for item in (
            f"{state}\t{district}",
            f"{target_state}\t{target_district}"
        )
    ]
)

# Create source state-district key and map it to the target state-district
demo_df = (
    demo_df
    .withColumn(
        "_state_district_key",
        F.concat_ws("\t", F.col("state"), F.col("district"))
    )
    .withColumn(
        "_mapped_state_district",
        mapping_expr[F.col("_state_district_key")]
    )
    .withColumn(
        "state",
        F.coalesce(
            F.split(F.col("_mapped_state_district"), "\t").getItem(0),
            F.col("state")
        )
    )
    .withColumn(
        "district",
        F.coalesce(
            F.split(F.col("_mapped_state_district"), "\t").getItem(1),
            F.col("district")
        )
    )
    .drop("_state_district_key", "_mapped_state_district")
)

# Re-check state-district combinations against the trusted reference dataset.
missing_districts = (
    demo_df
    .join(
        states_df.select("state", "district").distinct(),
        on=["state", "district"],
        how="left_anti"
    )
    .select("state", "district")
    .distinct()
)

display(missing_districts)


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3f037eec-0cfd-4ae7-8be4-dc21a98d3602)

In [17]:
bengaluru_df = demo_df.filter(demo_df["district"] == "bengaluru") \
              .select("state", "district", "pincode") \
              .distinct()

# Show the results
bengaluru_df.show()


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 19, Finished, Available, Finished, False)

+---------+---------+-------+
|    state| district|pincode|
+---------+---------+-------+
|karnataka|bengaluru| 560039|
|karnataka|bengaluru| 560052|
|karnataka|bengaluru| 560019|
|karnataka|bengaluru| 560069|
|karnataka|bengaluru| 560014|
|karnataka|bengaluru| 560028|
|karnataka|bengaluru| 560046|
+---------+---------+-------+



In [18]:
# map bengaluru district
demo_df = demo_df.withColumn(
    "district",
    when(substring(col("pincode").cast("string"), 1, 3) == "560", "bengaluru_urban")
    .when(substring(col("pincode").cast("string"), 1, 3).isin("561", "562" , "571"), "bengaluru_rural")
    .otherwise(col("district"))
)

StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 20, Finished, Available, Finished, False)

In [19]:
missing_states = demo_df.join(
    states_df, 
    on="state", 
    how="left_anti"
).select("state").distinct()

missing_states.show()


# Re-check state-district combinations against the trusted reference dataset.
missing_districts = (
    demo_df
    .join(
        states_df.select("state", "district").distinct(),
        on=["state", "district"],
        how="left_anti"
    )
    .select("state", "district")
    .distinct()
)

missing_districts.show()


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 21, Finished, Available, Finished, False)

+-----+
|state|
+-----+
+-----+

+-----+--------+
|state|district|
+-----+--------+
+-----+--------+



In [21]:

demo_df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("SILVER_Enriched_LAKEHOUSE.demographic")


StatementMeta(, ce79abaa-1754-4c38-bae8-be381ded481b, 24, Finished, Available, Finished, False)